# Track-A Experiment 6: QLoRA Fine-tuning of Qwen 2.5 7B

**Task**: SemEval-2026 Task 4 — Narrative Story Similarity (Track A)  
**Approach**: QLoRA fine-tuning for binary classification (text_a_is_closer: true/false)  
**Model**: Qwen/Qwen2.5-7B-Instruct  

## Setup
- Model loaded from HuggingFace Hub using Colab secrets token
- Data pulled from your HuggingFace dataset repo
- Fine-tuned model + results pushed back to HuggingFace

## 0. Configuration

**IMPORTANT**: Fill in your HuggingFace details below before running.

In [1]:
# ══════════════════════════════════════════════════════════════════════════
# FILL THESE IN
# ══════════════════════════════════════════════════════════════════════════

HF_TRAIN_REPO = "Chandan683/task4_syn_data"       # 1900 synthetic training examples
HF_DEV_REPO = "Chandan683/Dev_set_data"            # 200 validation examples
HF_TEST_REPO = "Chandan683/task4_test_data"        # 400 test examples
HF_MODEL_OUTPUT_REPO = "Chandan683/qwen2.5-7b-fewshot"  # where to push fine-tuned model
HF_RESULTS_REPO = "Chandan683/semeval-task4-resultsfewshot"  # where to push results

# Model
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Training hyperparameter
NUM_EPOCHS = 3
LEARNING_RATE = 1e-4
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

## 1. Install Dependencies

In [2]:
!pip install -q transformers>=4.44.0 datasets accelerate peft bitsandbytes trl huggingface_hub scikit-learn

## 2. Authenticate with HuggingFace

In [3]:
from google.colab import userdata
from huggingface_hub import login

# Reads the HF_TOKEN from Colab secrets
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
print("Logged in to HuggingFace.")

Logged in to HuggingFace.


## 3. Load Data from HuggingFace

In [4]:
from datasets import load_dataset

# Load training data (synthetic, 1900 examples)
train_dataset = load_dataset(HF_TRAIN_REPO)
print("Train dataset:", train_dataset)

# Load dev/validation data (200 examples)
dev_dataset = load_dataset(HF_DEV_REPO)
print("\nDev dataset:", dev_dataset)

# Load test data (400 examples, no labels)
test_dataset = load_dataset(HF_TEST_REPO)
print("\nTest dataset:", test_dataset)

print(f"\nTrain split: {train_dataset['train'].num_rows} examples")
print(f"Dev split:   {dev_dataset['validation'].num_rows} examples")
print(f"Test split:  {test_dataset['test'].num_rows} examples")

# Preview one training example
print("\n--- Sample training example ---")
print(train_dataset["train"][0])

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

synthetic_data_for_classification.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1900 [00:00<?, ? examples/s]

Train dataset: DatasetDict({
    train: Dataset({
        features: ['model_name', 'anchor_text', 'text_a', 'text_b', 'text_a_is_closer'],
        num_rows: 1900
    })
})


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

...


Dev dataset: DatasetDict({
    validation: Dataset({
        features: ['anchor_text', 'text_a', 'text_b', 'text_a_is_closer'],
        num_rows: 200
    })
})


README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

test_track_a.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]


Test dataset: DatasetDict({
    test: Dataset({
        features: ['anchor_text', 'text_a', 'text_b'],
        num_rows: 400
    })
})

Train split: 1900 examples
Dev split:   200 examples
Test split:  400 examples

--- Sample training example ---
{'model_name': 'meta-llama/Meta-Llama-3.1-8B-Instruct-Turbo', 'anchor_text': 'A mysterious individual, known only by their alias "The Wanderer", arrives in the small, rural town of Willow Creek, sparking curiosity and intrigue among its residents. The Wanderer, a woman with no discernible past or motivations, settles into a local boarding house and begins to integrate into the community. As the townspeople grow accustomed to her presence, they start to notice a series of subtle yet significant changes: abandoned buildings are repaired, neglected gardens are tended, and long-forgotten traditions are revived. The Wanderer\'s actions are met with a mix of gratitude and suspicion, with some viewing her as a benevolent force and others as a poten

## 4. Format Data for Instruction Tuning

Convert each triple into a chat-format prompt with the label as the response.

In [5]:
SYSTEM_PROMPT = """You are an expert in narrative analysis. Your task is to determine which of two stories is more narratively similar to an anchor story.

Narrative similarity is based on three components:
1. Abstract Theme: The underlying ideas and motives of the story.
2. Course of Action: The sequence of central events and turning points.
3. Outcomes: The results and conclusions of the story.

Do NOT focus on surface-level similarities like setting, time period, character names, or genre. Focus on the deeper narrative structure.

Here are some labelled examples:

--- Example 1 ---
### Anchor Story
The film takes place in Burma and India during World War II.
A British officer falls in love with his Japanese instructor at a military language school. They start a romance, but she is regarded as the enemy and is not accepted by his countrymen. They marry in secret and plan on spending his two weeks' leave together. When one of the other officers is injured, he is sent into the field as an interrogator. Later he is captured by the Japanese army when he is patrolling with a brigadier and an Indian driver in a Japanese-controlled zone. He escapes and returns to his own lines, only to discover that his wife is suffering from a brain tumour. Although the doctor initially gives her good odds of surviving, she dies after an operation.

### Text A
During the Irish War of Independence in 1921, Irish rebel leader Dennis Riordan (Aherne) and English aristocrat Helen Drummond (Oberon) meet and fall in love. Riordan is pursued, however, by British army officer Captain Preston (Niven).
The original film ended with Riordan getting shot and killed, but did not do well at the box office. A happier ending was also filmed which has Riordan being shot but surviving; all subsequent versions have. The original cut has since been lost.
The movie has several comic relief scenes: after a raid on an IRA "safe house", British officers grumble about being not being able to find Riordan, who is in fact standing just behind them; when the Irish Delegation goes to a formal ball and is asked by the footman for their names to be announced, the delegation replies in Irish.

### Text B
1914, German advance through Belgium: the young war volunteer Alexander 'Alex' Haller (Schell) is given water by an equally young Belgian woman (Berger).
1917, Third Battle of Flanders: Alex, now a 2nd lieutenant, is tired about the propaganda at the Home Front, so he spends his furlough in the hinterland of the Western Front.
While boarding in a brothel, he meets the young woman again. They fall in love.
Late 1918, German retreat after the Armistice: Engele and Alex meet again only to be harassed by a Belgian mob. Shortly prior to be hanged by the mob, a group of passing Belgian soldiers, tired about killing, saves them. End of the tale.

Answer: B

--- Example 2 ---
### Anchor Story
The foursome (Gérard Rinaldi, Jean Sarrus, Gérard Filipelli, Jean-Guy Fechner) are on a holiday. The Little Olympic flame is to be passed through their village. A grocer (Paul Préboist) calls upon them for help in decorating the village. On their job Gérard falls for the grocer's daughter Délice (Martine Kelly). However she runs away with the sportsman with the flame. The four then enter the Little Olympics to try to win her back and cause havoc in the process.

### Text A
The old grandmother Tina arrives in town to attend the wedding of his nephew Alberto with his girlfriend Ileana.
Upon arrival she discovers that she has been stolen of a medallion that her late husband had given her.
He goes to the police station to file a complaint and get the dear object back, but given the length of the investigation, he decides to carry out the search for the thief himself, combining a great deal of mess.
Eventually, by chance, he finds the thief, who lives in the same hotel, also managing to have an entire gang of criminals arrested.
The grandson Alberto can marry the beautiful Ileana and the grandmother Tina will be appointed, by merit, an honorary colonel of the female police.

### Text B
Brendan Byers III is a rich playboy who enlists to fight in the war against the Axis powers, but is classified 4-F. He really wants to fight, so he enlists other 4-Fs and some loyal volunteers from his own service staff and forms his own army, financing their training and equipment. Once completed, they travel to the front in Italy, with Byers impersonating a Nazi general named Eric Kesselring.
The plan is to pull back the German lines, since the front has remained static for too long, enabling the Allies to push forward again. The mission does not go smoothly and they must overcome several obstacles, including the fiery wife of the local mayor who is the real Kesselring's lover, and the real Kesselring's involvement in an assassination attempt on Hitler. Afterwards, they face their next mission: infiltrating the Imperial Japanese command to influence the outcome of the Battle of Kwajalein.

Answer: A

--- Example 3 ---
### Anchor Story
A teenage girl accuses her primary schoolteacher, Jean Doucet (Jacques Brel), of trying to rape her. The police and the mayor investigate, but Doucet denies the charges. Two other students come forward to reveal more of Doucet's misconduct – one confessing to be his mistress. Doucet faces trial and hard labour if convicted.

### Text A
Art Brooks and Kelly Moore are a couple who get married. But Kelly realizes that Art is an alcoholic, and he beats her. She starts a relationship with the lawyer Richard Linsky. Art is then found dead, with Richard being convicted and imprisoned.

### Text B
Tonino is a high school student, in love with the new teacher of letters, Mrs. Moretti. One day Tonino and the teacher are kidnapped and locked up in a trullo where they consummate their love. Later, Moretti is transferred to Rome, and a new and beautiful teacher is assigned to Toninos class.

Answer: B

--- Now classify this ---
Given the anchor story and two candidate stories (Text A and Text B), determine which candidate is more narratively similar to the anchor.

Respond with ONLY "A" or "B" (nothing else)."""


def format_as_chat(example):
    """Convert a Track-A example into chat messages for instruction tuning."""
    user_msg = (
        f"### Anchor Story\n{example['anchor_text']}\n\n"
        f"### Text A\n{example['text_a']}\n\n"
        f"### Text B\n{example['text_b']}\n\n"
        f"Which text is more narratively similar to the anchor?"
    )

    label = "A" if example["text_a_is_closer"] else "B"

    example["messages"] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": label},
    ]
    return example


def format_as_chat_inference(example):
    """Convert a Track-A example into chat messages for inference (no label)."""
    user_msg = (
        f"### Anchor Story\n{example['anchor_text']}\n\n"
        f"### Text A\n{example['text_a']}\n\n"
        f"### Text B\n{example['text_b']}\n\n"
        f"Which text is more narratively similar to the anchor?"
    )

    example["messages"] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]
    return example


# Filter out rows with null text fields from synthetic training data
train_raw = train_dataset["train"].filter(
    lambda x: x["anchor_text"] is not None and x["text_a"] is not None and x["text_b"] is not None
)
train_data = train_raw.map(format_as_chat)

# Dev data (full 200 examples with labels)
dev_data = dev_dataset["validation"]

# Test data (no labels)
test_data = test_dataset["test"].map(format_as_chat_inference)

print(f"Training examples: {len(train_data)} (after filtering nulls from {train_dataset['train'].num_rows})")
print(f"Dev examples:      {len(dev_data)}")
print(f"Test examples:     {len(test_data)}")
print(f"\n--- Formatted training example ---")
print(train_data[0]["messages"])

Filter:   0%|          | 0/1900 [00:00<?, ? examples/s]

Map:   0%|          | 0/1897 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Training examples: 1897 (after filtering nulls from 1900)
Dev examples:      200
Test examples:     400

--- Formatted training example ---
[{'content': 'You are an expert in narrative analysis. Your task is to determine which of two stories is more narratively similar to an anchor story.\n\nNarrative similarity is based on three components:\n1. Abstract Theme: The underlying ideas and motives of the story.\n2. Course of Action: The sequence of central events and turning points.\n3. Outcomes: The results and conclusions of the story.\n\nDo NOT focus on surface-level similarities like setting, time period, character names, or genre. Focus on the deeper narrative structure.\n\nHere are some labelled examples:\n\n--- Example 1 ---\n### Anchor Story\nThe film takes place in Burma and India during World War II.\nA British officer falls in love with his Japanese instructor at a military language school. They start a romance, but she is regarded as the enemy and is not accepted by his country

## 5. Load Model with QLoRA (4-bit quantization)

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    token=hf_token,
    trust_remote_code=True,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

print(f"Model loaded: {BASE_MODEL}")
print(f"Model dtype: {model.dtype}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2.5-7B-Instruct
Model dtype: torch.float32
GPU memory used: 7.7 GB


## 6. Configure LoRA

In [7]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 7. Train with SFTTrainer

In [8]:
from trl import SFTTrainer, SFTConfig

tokenizer.model_max_length = MAX_SEQ_LENGTH

training_args = SFTConfig(
    output_dir="./results_qwen_qlora",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

Tokenizing train dataset:   0%|          | 0/1897 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2114 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/1897 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


Step,Training Loss
10,2.804300
20,2.452700
30,1.460400
40,0.134200


Step,Training Loss
10,2.804300
20,2.452700
30,1.460400
40,0.134200
50,0.005300
60,0.000300
70,0.000200
80,0.000100
90,0.000100
100,0.000100


Training complete!


## 8. Evaluate on Dev Set

In [9]:
from sklearn.metrics import accuracy_score, classification_report
import json


def predict_single(messages, model, tokenizer):
    """Run inference on a single example and return predicted label."""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8,
            temperature=0.01,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the generated tokens
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True).strip().upper()

    if response.startswith("A"):
        return True
    elif response.startswith("B"):
        return False
    else:
        print(f"  Unparseable: '{response}', defaulting to A")
        return True  # fallback


# Evaluate on full dev set (Chandan683/Dev_set_data, 200 examples)
dev_eval = dev_data.map(format_as_chat_inference)

gold_labels = []
pred_labels = []
predictions = []

print(f"Evaluating on {len(dev_eval)} dev examples...")
for i, example in enumerate(dev_eval):
    pred = predict_single(example["messages"], model, tokenizer)
    gold = dev_data[i]["text_a_is_closer"]

    gold_labels.append(gold)
    pred_labels.append(pred)
    predictions.append({
        "id": i,
        "gold": gold,
        "pred": pred,
        "correct": gold == pred,
    })

    if (i + 1) % 20 == 0:
        current_acc = sum(p["correct"] for p in predictions) / len(predictions)
        print(f"  [{i+1}/{len(dev_eval)}] running accuracy: {current_acc:.4f}")

accuracy = accuracy_score(gold_labels, pred_labels)
print(f"\n{'='*60}")
print(f"Dev Accuracy: {accuracy:.4f} ({sum(g == p for g, p in zip(gold_labels, pred_labels))}/{len(gold_labels)})")
print(f"\nClassification Report:")
print(classification_report(gold_labels, pred_labels, target_names=["B (text_b)", "A (text_a)"]))

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


Evaluating on 200 dev examples...
  [20/200] running accuracy: 0.6500
  [40/200] running accuracy: 0.5500
  [60/200] running accuracy: 0.5167
  [80/200] running accuracy: 0.5750
  [100/200] running accuracy: 0.5800
  [120/200] running accuracy: 0.5667
  [140/200] running accuracy: 0.6071
  [160/200] running accuracy: 0.6062
  [180/200] running accuracy: 0.5833
  [200/200] running accuracy: 0.5750

Dev Accuracy: 0.5750 (115/200)

Classification Report:
              precision    recall  f1-score   support

  B (text_b)       0.57      0.55      0.56        99
  A (text_a)       0.58      0.60      0.59       101

    accuracy                           0.57       200
   macro avg       0.57      0.57      0.57       200
weighted avg       0.57      0.57      0.57       200



## 9. Generate Test Predictions

In [10]:
test_predictions = []

print(f"Generating predictions for {len(test_data)} test examples...")
for i, example in enumerate(test_data):
    pred = predict_single(example["messages"], model, tokenizer)
    test_predictions.append({"text_a_is_closer": pred})

    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/{len(test_data)}] processed.")

# Save as JSONL (submission format)
with open("track_a.jsonl", "w") as f:
    for pred in test_predictions:
        f.write(json.dumps(pred) + "\n")

print(f"\nTest predictions saved to track_a.jsonl")
print(f"Total predictions: {len(test_predictions)}")
print(f"Label distribution: {sum(p['text_a_is_closer'] for p in test_predictions)} × A, "
      f"{sum(not p['text_a_is_closer'] for p in test_predictions)} × B")

Generating predictions for 400 test examples...
  [20/400] processed.
  [40/400] processed.
  [60/400] processed.
  [80/400] processed.
  [100/400] processed.
  [120/400] processed.
  [140/400] processed.
  [160/400] processed.
  [180/400] processed.
  [200/400] processed.
  [220/400] processed.
  [240/400] processed.
  [260/400] processed.
  [280/400] processed.
  [300/400] processed.
  [320/400] processed.
  [340/400] processed.
  [360/400] processed.
  [380/400] processed.
  [400/400] processed.

Test predictions saved to track_a.jsonl
Total predictions: 400
Label distribution: 215 × A, 185 × B


## 10. Save Dev Evaluation Results

In [11]:
results = {
    "experiment": "track_a_exp6_qlora_finetune",
    "base_model": BASE_MODEL,
    "method": "QLoRA",
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dev_accuracy": accuracy,
    "dev_correct": int(sum(g == p for g, p in zip(gold_labels, pred_labels))),
    "dev_total": len(gold_labels),
}

with open("results_exp6.json", "w") as f:
    json.dump(results, f, indent=2)

# Save per-example predictions
with open("dev_predictions.jsonl", "w") as f:
    for pred in predictions:
        f.write(json.dumps(pred) + "\n")

print("Results saved locally:")
print("  - results_exp6.json")
print("  - dev_predictions.jsonl")
print("  - track_a.jsonl")

Results saved locally:
  - results_exp6.json
  - dev_predictions.jsonl
  - track_a.jsonl


## 11. Push Fine-tuned Model to HuggingFace

In [12]:
# Save and push the LoRA adapter
model.save_pretrained("./qwen25_7b_lora_adapter")
tokenizer.save_pretrained("./qwen25_7b_lora_adapter")

model.push_to_hub(HF_MODEL_OUTPUT_REPO, token=hf_token)
tokenizer.push_to_hub(HF_MODEL_OUTPUT_REPO, token=hf_token)

print(f"Model pushed to: https://huggingface.co/{HF_MODEL_OUTPUT_REPO}")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 46.2kB / 80.8MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpbutzuayd/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

No files have been modified since last commit. Skipping to prevent empty commit.


Model pushed to: https://huggingface.co/Chandan683/qwen2.5-7b-narrative-similarity


## 12. Push Results to HuggingFace

In [13]:
from huggingface_hub import HfApi

api = HfApi()

# Create the results repo if it doesn't exist
try:
    api.create_repo(HF_RESULTS_REPO, repo_type="dataset", exist_ok=True, token=hf_token)
except Exception as e:
    print(f"Repo creation note: {e}")

# Upload results files
for filename in ["results_exp6.json", "dev_predictions.jsonl", "track_a.jsonl"]:
    api.upload_file(
        path_or_fileobj=filename,
        path_in_repo=f"track_a_exp6_qlora/{filename}",
        repo_id=HF_RESULTS_REPO,
        repo_type="dataset",
        token=hf_token,
    )
    print(f"  Uploaded: {filename}")

print(f"\nResults pushed to: https://huggingface.co/datasets/{HF_RESULTS_REPO}")

  Uploaded: results_exp6.json
  Uploaded: dev_predictions.jsonl
  Uploaded: track_a.jsonl

Results pushed to: https://huggingface.co/datasets/Chandan683/semeval-task4-results


## 13. Summary

In [14]:
print("=" * 60)
print("EXPERIMENT 6 SUMMARY")
print("=" * 60)
print(f"Base model:       {BASE_MODEL}")
print(f"Method:           QLoRA (4-bit NF4)")
print(f"LoRA rank:        {LORA_R}")
print(f"Training data:    {len(train_data)} examples (from {HF_TRAIN_REPO})")
print(f"Dev data:         {len(dev_data)} examples (from {HF_DEV_REPO})")
print(f"Test data:        {len(test_data)} examples (from {HF_TEST_REPO})")
print(f"Epochs:           {NUM_EPOCHS}")
print(f"Learning rate:    {LEARNING_RATE}")
print(f"Eff. batch size:  {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Dev accuracy:     {accuracy:.4f}")
print(f"Test predictions: {len(test_predictions)}")
print(f"\nModel:   https://huggingface.co/{HF_MODEL_OUTPUT_REPO}")
print(f"Results: https://huggingface.co/datasets/{HF_RESULTS_REPO}")

EXPERIMENT 6 SUMMARY
Base model:       Qwen/Qwen2.5-7B-Instruct
Method:           QLoRA (4-bit NF4)
LoRA rank:        16
Training data:    1897 examples (from Chandan683/task4_syn_data)
Dev data:         200 examples (from Chandan683/Dev_set_data)
Test data:        400 examples (from Chandan683/task4_test_data)
Epochs:           3
Learning rate:    0.0001
Eff. batch size:  16
Dev accuracy:     0.5750
Test predictions: 400

Model:   https://huggingface.co/Chandan683/qwen2.5-7b-narrative-similarity
Results: https://huggingface.co/datasets/Chandan683/semeval-task4-results
